In [ ]:
import os
# =====================
# 1️⃣ Import libraries
# =====================
from chromadb.config import Settings
import chromadb
from sentence_transformers import SentenceTransformer, util
from collections import Counter
import ipywidgets as widgets
from IPython.display import display, clear_output
import torch

# =====================
# 2️⃣ เชื่อมต่อกับ ChromaDB
# =====================
client = chromadb.HttpClient(
    host=os.getenv("CHROMA_HOST", "localhost"),  # เปลี่ยนเป็น host จริง
    port=int(os.getenv("CHROMA_PORT", "8000")),               # port จริง
    settings=Settings(anonymized_telemetry=False)
)
collection = client.get_collection("jobs_search001")  # ชื่อ collection จริง

# =====================
# 3️⃣ ดึงข้อมูลจาก collection
# =====================
# include=["metadatas"] พอเพราะเราต้องการชื่อบริษัท
all_data = collection.get(include=["documents", "metadatas"])

company_names = []
metadatas = []

for doc, meta in zip(all_data["documents"], all_data["metadatas"]):
    # ใช้ company_name ภาษาไทยก่อน ถ้าไม่มีใช้ภาษาอังกฤษ
    name = meta.get("company_name") or meta.get("company_name_eng") or ""
    if name:  # กรอง empty string
        company_names.append(name)
        metadatas.append(meta)

# =====================
# 4️⃣ Load embedding model
# =====================
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

company_embeddings = model.encode(company_names, convert_to_tensor=True)
# normalize embeddings เพื่อความแม่นยำ
company_embeddings = torch.nn.functional.normalize(company_embeddings, p=2, dim=1)

# =====================
# 5️⃣ Popularity counter
# =====================
# กำหนดเป็น 1 หากไม่มี search logs
counter = Counter({name: 1 for name in company_names})

# =====================
# 6️⃣ Semantic autocomplete function
# =====================
def autocomplete_company(q, limit=5):
    if not q:
        return []
    
    query_emb = model.encode(q, convert_to_tensor=True)
    query_emb = torch.nn.functional.normalize(query_emb, p=2, dim=0)
    
    # semantic similarity
    scores = util.cos_sim(query_emb, company_embeddings)[0]
    
    results_list = []
    for idx, name in enumerate(company_names):
        score = float(scores[idx]) + (counter[name] * 0.1)
        meta = metadatas[idx]  # ✅ ใช้ metadata ที่ดึงจาก collection จริง
        results_list.append((name, score, meta))
    
    # sort by score descending
    results_list = sorted(results_list, key=lambda x: x[1], reverse=True)
    
    return results_list[:limit]

# =====================
# 7️⃣ Interactive widget (ใช้ Combobox)
# =====================
combo = widgets.Combobox(
    placeholder='พิมพ์ชื่อบริษัท...',
    options=[],          # จะอัปเดตเมื่อพิมพ์
    description='Company:',
    ensure_option=False
)

output = widgets.Output()

def on_combo_change(change):
    with output:
        clear_output(wait=True)
        q = change['new']
        if q:
            suggestions = autocomplete_company(q)
            # update Combobox options
            combo.options = [name for name, _, _ in suggestions]

            print("👉 คำแนะนำ:")
            for name, score, meta in suggestions:
                jobpost_id = meta.get('jobpost_id', 'N/A')
                company_id = meta.get('company_id', 'N/A')
                print(f"- {name} (jobpost_id={jobpost_id}, company_id={company_id}) [score={score:.3f}]")

combo.observe(on_combo_change, names='value')
display(combo, output)


In [ ]:
for meta in metadatas[:5]:
    print(meta.keys())


In [ ]:
results = collection.get(include=["metadatas", "documents", "embeddings"])
print("Documents sample:", results["documents"][:5])
print("Metadatas sample:", results["metadatas"][:5])


In [ ]:
import os
import chromadb
from chromadb.config import Settings

# ------------------ 1. เชื่อมต่อ ChromaDB ------------------
client = chromadb.HttpClient(
    host=os.getenv("CHROMA_HOST", "localhost"),
    port=int(os.getenv("CHROMA_PORT", "8000")),
    settings=Settings(anonymized_telemetry=False)
)

# ดู collections
collections = client.list_collections()
print(f"Collections: {[c.name for c in collections]}")

In [ ]:
results = collection.get(include=["metadatas", "documents"])
print("ตัวอย่าง metadata 5 แถวแรก:")
for i, m in enumerate(results["metadatas"][:5]):
    print(i, m)
